# Prosperity Initial Data Loading

In [12]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import plotly.express as px
from scipy.stats import norm
import glob
import os

ROUND = 4

# Constants
DAY_OFFSET = 1000000
DATA_DIR = f"ROUND_{ROUND}"

# Days & TTE for options
DAYS = [0, 1, 2, 3]

In [13]:
def load_round_data(base_path, round_num):
    all_prices = []
    all_trades = []
    
    # 1. Automatically find all price files for this round
    price_pattern = os.path.join(base_path, f"prices_round_{round_num}_day_*.csv")
    price_files = sorted(glob.glob(price_pattern))
    
    # 2. Extract and sort day numbers to ensure the timeline is in order
    
    print(f"Loading Round {round_num} data from {base_path} for days: {DAYS}")
    
    for i, day in enumerate(DAYS):
        # Load Prices
        price_file = os.path.join(base_path, f"prices_round_{round_num}_day_{day}.csv")
        if os.path.exists(price_file):
            df_p = pd.read_csv(price_file, sep=";")
            df_p['timestamp']
            df_p['absolute_timestamp'] = df_p['timestamp'] + (i * DAY_OFFSET)
            df_p['TTE'] = 8 - day
            all_prices.append(df_p)
            
        # Load Trades
        trade_file = os.path.join(base_path, f"trades_round_{round_num}_day_{day}.csv")
        if os.path.exists(trade_file):
            df_t = pd.read_csv(trade_file, sep=";")
            df_t['timestamp']
            df_t['absolute_timestamp'] = df_t['timestamp'] + (i * DAY_OFFSET)
            df_t['day'] = day
            all_trades.append(df_t)
            
    if not all_prices:
        raise ValueError(f"No files found for Round {round_num} at {base_path}")

    prices_df = pd.concat(all_prices, ignore_index=True)
    trades_df = pd.concat(all_trades, ignore_index=True) if all_trades else pd.DataFrame()
    
    return prices_df, trades_df

# Load the data using the ROUND variable
prices, trades = load_round_data(DATA_DIR, ROUND)

# Cleaning logic
prices['mid_price'] = prices['mid_price'].replace(0, np.nan)
prices['mid_price'] = prices.groupby('product')['mid_price'].transform(lambda x: x.interpolate())

print(f"Total: Loaded {len(prices)} price points and {len(trades)} trades.")

Loading Round 4 data from ROUND_4 for days: [0, 1, 2, 3]
Total: Loaded 480000 price points and 5589 trades.


In [14]:
def visualize_product(product_name, prices_df, trades_df, y_start=None, y_end=None, regression=False, sma_window=None):
    p_df = prices_df[prices_df['product'] == product_name].sort_values('absolute_timestamp')
    t_df = trades_df[trades_df['symbol'] == product_name].sort_values('absolute_timestamp')
    
    fig = go.Figure()

    # 1. Bid/Ask Shading (Spread)
    fig.add_trace(go.Scatter(
        x=pd.concat([p_df['absolute_timestamp'], p_df['absolute_timestamp'][::-1]]),
        y=pd.concat([p_df['ask_price_1'], p_df['bid_price_1'][::-1]]),
        fill='toself',
        fillcolor='rgba(100, 100, 100, 0.2)',
        line=dict(color='rgba(255,255,255,0)'),
        hoverinfo="skip",
        name='Spread (Ask1-Bid1)'
    ))

    # 2. Mid Price Line
    fig.add_trace(go.Scatter(
        x=p_df['absolute_timestamp'],
        y=p_df['mid_price'],
        mode='lines',
        name='Mid Price',
        line=dict(color='#1f77b4', width=2)
    ))
    
    # 3. Linear Regression Overlay
    if regression:
        x = p_df['absolute_timestamp']
        y = p_df['mid_price']
        m, b = np.polyfit(x, y, 1)
        reg_line = m * x + b
        fig.add_trace(go.Scatter(x=x, y=reg_line, mode='lines', name=f'Trend (slope={m:.4f})', 
                                 line=dict(color='#d62728', width=2, dash='dash')))

    # 4. Moving Average Overlay
    if sma_window:
        sma = p_df['mid_price'].rolling(window=sma_window).mean()
        fig.add_trace(go.Scatter(x=p_df['absolute_timestamp'], y=sma, mode='lines', name=f'{sma_window}-Tick SMA', 
                                 line=dict(color='#bcbd22', width=2)))

    # 5. Trades Scatter
    if not t_df.empty:
        fig.add_trace(go.Scatter(
            x=t_df['absolute_timestamp'],
            y=t_df['price'],
            mode='markers',
            name='Trades',
            marker=dict(size=6, color='#ff7f0e', symbol='diamond', line=dict(width=1, color='white')),
            text=t_df['quantity'].apply(lambda q: f"Qty: {q}")
        ))

    fig.update_layout(
        title=f"Market Analysis: {product_name}",
        xaxis_title="Timestamp",
        yaxis_title="Price",
        template="plotly_dark",
        hovermode="x unified",
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
    )
    
    for i in range(1, 3):
        fig.add_vline(x=i * DAY_OFFSET, line_dash="dash", line_color="gray", annotation_text=f"Day {i-2} -> {i-1}")
    
    if y_start is not None and y_end is not None:
        fig.update_yaxes(range=[y_start, y_end])

    return fig


In [15]:

products = ['VELVETFRUIT_EXTRACT', 'HYDROGEL_PACK']

visualize_product('VELVETFRUIT_EXTRACT', prices, trades).show()
visualize_product('HYDROGEL_PACK', prices, trades).show()

stats = []
for product in prices['product'].unique():
    p_df = prices[prices['product'] == product]
    t_df = trades[trades['symbol'] == product]
    
    stats.append({
        'Product': product,
        'Avg Mid Price': p_df['mid_price'].mean(),
        'Std Dev': p_df['mid_price'].std(),
        'Avg Spread': (p_df['ask_price_1'] - p_df['bid_price_1']).mean(),
        'Total Trade Volume': t_df['quantity'].sum(),
        'Trade Count': len(t_df)
    })

pd.DataFrame(stats).set_index('Product')

,Avg Mid Price,Std Dev,Avg Spread,Total Trade Volume,Trade Count
Product,,,,,
VEV_5400,14.089150,4.608139,1.335450,1177,340
VEV_6500,0.500000,0.000000,1.000000,1425,408
VEV_5500,5.545575,2.476997,1.126750,1350,387
VEV_5200,91.112837,12.796427,2.799875,177,50
VEV_5300,43.105350,8.975917,2.019950,676,201
VELVETFRUIT_EXTRACT,5247.364100,17.090769,4.985200,10892,1826
VEV_4500,747.372962,17.104594,15.791425,6,3
HYDROGEL_PACK,9993.731250,32.588283,15.721200,5445,1346
VEV_5000,251.673313,16.381336,5.973625,6,3
